# Exercise 5 — Full Pipeline: Indicators → Signal → Backtest

This exercise connects the full Day 89–91 stack: synthetic OHLCV → `add_indicators` → RSI-based signal → `run_backtest` → metrics. The signal rule: go long (1) when RSI < 40 (oversold), go flat (0) when RSI > 60 (overbought), otherwise hold the previous position.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def sma(s, w=20):  return s.rolling(window=w).mean()
def ema(s, w=20):  return s.ewm(span=w, adjust=False).mean()
def rsi(s, w=14):
    import warnings
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))
def macd(s, fast=12, slow=26, sig=9):
    ml = ema(s, fast) - ema(s, slow)
    sl = ema(ml, sig)
    return pd.DataFrame({"macd": ml, "signal": sl, "histogram": ml - sl})
def bollinger_bands(s, w=20, ns=2.0):
    mid = sma(s, w); std = s.rolling(w).std()
    return pd.DataFrame({"upper": mid+ns*std, "middle": mid, "lower": mid-ns*std})
def add_indicators(df, sw=20, ew=20, rw=14, mf=12, ms=26, mg=9, bw=20, bs=2.0):
    df = df.copy(); c = df["Close"]
    df[f"sma_{sw}"] = sma(c, sw); df[f"ema_{ew}"] = ema(c, ew)
    df[f"rsi_{rw}"] = rsi(c, rw)
    m = macd(c, mf, ms, mg)
    df["macd"] = m["macd"]; df["macd_signal"] = m["signal"]; df["macd_hist"] = m["histogram"]
    bb = bollinger_bands(c, bw, bs)
    df["bb_upper"] = bb["upper"]; df["bb_middle"] = bb["middle"]; df["bb_lower"] = bb["lower"]
    return df
def compute_returns(df):
    return df["Close"].pct_change()

def compute_equity(returns, initial=1.0):
    return (1 + returns.fillna(0)).cumprod() * initial
def max_drawdown(equity):
    peak = equity.cummax()
    return float(((equity - peak) / peak).min())
def sharpe_ratio(returns, periods_per_year=252):
    clean = returns.dropna()
    if len(clean) == 0 or clean.std() == 0:
        return 0.0
    return float(clean.mean() / clean.std() * (periods_per_year ** 0.5))
def run_backtest(df, signals):
    market_returns   = compute_returns(df)
    positions        = signals.shift(1).fillna(0)
    strategy_returns = positions * market_returns
    equity  = compute_equity(strategy_returns)
    clean   = strategy_returns.dropna()
    n_days  = len(clean)
    win_rate = float((clean > 0).sum() / max(n_days, 1))
    pos_diff = positions.diff().fillna(0)
    n_trades = int((pos_diff != 0).sum())
    total_ret = float(equity.iloc[-1] - 1.0)
    base = 1.0 + total_ret
    ann_ret = float(base ** (252.0 / max(n_days, 1)) - 1) if base > 0 else -1.0
    return {
        "total_return":      total_ret,
        "annualized_return": ann_ret,
        "sharpe_ratio":      sharpe_ratio(strategy_returns),
        "max_drawdown":      max_drawdown(equity),
        "win_rate":          win_rate,
        "n_trades":          n_trades,
        "equity":            equity,
        "strategy_returns":  strategy_returns,
        "market_returns":    market_returns,
    }

# ── Exercise: build and run a complete RSI mean-reversion strategy ────────────

def make_rsi_signal(enriched, oversold=40, overbought=60):
    """RSI mean-reversion signal.

    Rules:
      - Long (1) when RSI < oversold
      - Flat (0) when RSI > overbought
      - Otherwise: hold the previous position (forward-fill)

    Args:
        enriched   : DataFrame with "rsi_14" column (output of add_indicators)
        oversold   : RSI level to go long (default 40)
        overbought : RSI level to go flat (default 60)

    Returns:
        pd.Series of {0, 1} aligned to enriched.index
    """
    # TODO:
    # 1. signal = pd.Series(float("nan"), index=enriched.index)
    # 2. signal[enriched["rsi_14"] < oversold]  = 1.0
    # 3. signal[enriched["rsi_14"] > overbought] = 0.0
    # 4. signal = signal.ffill().fillna(0.0)  — carry forward; default flat
    # 5. return signal
    return pd.Series(0.0, index=enriched.index)


### Checks

In [ ]:
checks = 0

# 1 — make_rsi_signal returns a Series aligned to df
try:
    df       = _synthetic()
    enriched = add_indicators(df)
    sig      = make_rsi_signal(enriched)
    assert isinstance(sig, pd.Series) and len(sig) == len(enriched)
    checks += 1; print("✅ 1 make_rsi_signal returns same-length Series")
except Exception as e:
    print("❌ 1:", e)

# 2 — signal values are only 0 or 1
try:
    df       = _synthetic()
    enriched = add_indicators(df)
    sig      = make_rsi_signal(enriched)
    valid    = set(sig.unique()) - {0.0, 1.0, 0, 1}
    assert not valid, f"unexpected signal values: {valid}"
    checks += 1; print("✅ 2 signal values are only 0 and 1")
except Exception as e:
    print("❌ 2:", e)

# 3 — when RSI < 40, signal should be 1
try:
    df       = _synthetic()
    enriched = add_indicators(df)
    sig      = make_rsi_signal(enriched, oversold=40, overbought=60)
    oversold_mask = enriched["rsi_14"] < 40
    if oversold_mask.any():
        assert (sig[oversold_mask] == 1.0).all(), "oversold RSI should give signal=1"
    checks += 1; print("✅ 3 RSI < 40 → signal = 1")
except Exception as e:
    print("❌ 3:", e)

# 4 — run_backtest accepts the signal and returns valid metrics
try:
    df       = _synthetic()
    enriched = add_indicators(df)
    sig      = make_rsi_signal(enriched)
    result   = run_backtest(df, sig)
    assert isinstance(result["total_return"], float)
    assert isinstance(result["equity"], pd.Series)
    assert len(result["equity"]) == len(df)
    checks += 1; print("✅ 4 run_backtest runs on RSI signal without error")
except Exception as e:
    print("❌ 4:", e)

# 5 — print metrics summary
try:
    df       = _synthetic()
    enriched = add_indicators(df)
    sig      = make_rsi_signal(enriched)
    r        = run_backtest(df, sig)
    print(f"\n  RSI strategy metrics:")
    print(f"    Total return    : {r['total_return']:.2%}")
    print(f"    Annualised ret  : {r['annualized_return']:.2%}")
    print(f"    Sharpe ratio    : {r['sharpe_ratio']:.3f}")
    print(f"    Max drawdown    : {r['max_drawdown']:.2%}")
    print(f"    Win rate        : {r['win_rate']:.2%}")
    print(f"    Trades          : {r['n_trades']}")
    checks += 1; print("✅ 5 metrics printed successfully")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
